In [1]:
from datetime import datetime, timezone
from src.helpers import get_project_folders, normalize_project
from src.metric import MonthRatioMetric, MonthMetric
from pathlib import Path
import orjson

In [2]:
INPUT_FOLDER = "/home/jortvd/thesis-data/code-results-9-4-2026"
OUTPUT_FOLDER = "../results"

In [3]:
def is_code_file(file: Path):
    return file.suffix in [
        ".rs", # Rust
        ".java", # Java
        ".cpp", ".c", ".h", ".hpp", # C/C++
        ".js", ".ts", ".tsx", ".jsx", ".vue", ".svelte", # JavaScript/TypeScript
        ".go", # Go
        ".py", ".pyx", ".pxd", # Python/Cython
        ".rb", # Ruby
        ".swift", # Swift
        ".kt", ".kts", # Kotlin
        ".scala", # Scala
        ".hs", # Haskell
    ]

def is_rust_file(file: Path):
    return file.suffix == ".rs"

In [5]:
metric = MonthRatioMetric("churn_per_month", rust_split=True)

for project_folder in get_project_folders(INPUT_FOLDER):
    with open(project_folder / "commit_stats.json") as f:
        commits = orjson.loads(f.read())
    print(f"Processing {project_folder.name} with {len(commits)} commits")
    
    commits = sorted(commits, key=lambda c: c["author_date"])
    rust_lines = 0
    other_lines = 0

    for commit in commits:
        date = datetime.fromtimestamp(commit["author_date"], tz=timezone.utc)
        rust_churn = 0
        other_churn = 0

        for file in commit["file_details"]:
            if not is_code_file(Path(file["new_path"])):
                continue
            if not "lines_added" in file or not "lines_deleted" in file:
                continue
            
            if is_rust_file(Path(file["new_path"])):
                rust_churn += file["lines_added"] + file["lines_deleted"]
                rust_lines += file["lines_added"] - file["lines_deleted"]
            else:
                other_churn += file["lines_added"] + file["lines_deleted"]
                other_lines += file["lines_added"] - file["lines_deleted"]
        
        metric.add_num(normalize_project(project_folder.name), date, rust_churn, is_rust=True)
        metric.add_num(normalize_project(project_folder.name), date, other_churn, is_rust=False)
        metric.set_den(normalize_project(project_folder.name), date, rust_lines, is_rust=True)
        metric.set_den(normalize_project(project_folder.name), date, other_lines, is_rust=False)

metric.save(OUTPUT_FOLDER)

Processing BitBoxSwiss_bitbox02-firmware with 1428 commits
Processing KiwiTalk_KiwiTalk with 1887 commits
Processing NixOS_nixos-search with 598 commits
Processing OISF_suricata with 18689 commits
Processing Pometry_Raphtory with 1342 commits
Processing SeaDve_Kooha with 1980 commits
Processing amir20_dtop with 453 commits
Processing apache_arrow with 18674 commits
Processing apache_arrow-rs with 7483 commits
Processing appsinacup_godot-rapier-physics with 315 commits
Processing ariebovenberg_whenever with 402 commits
Processing brndnmtthws_dryoc with 382 commits
Processing cjdelisle_cjdns with 2883 commits
Processing cooklang_cookcli with 317 commits
Processing coreboot_coreboot with 62653 commits
Processing emmericp_ixy with 50 commits
Processing eraserhd_parinfer-rust with 584 commits
Processing fosskers_aura with 1832 commits
Processing gschup_ggrs with 623 commits
Processing intentee_paddler with 334 commits
Processing ixy-languages_ixy.rs with 99 commits
Processing jedisct1_libso